# Capítulo 7 · Medidas de associação e impacto em saúde: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, um registro de 2.400 pacientes de cirurgia eletiva por trás do rascunho de um resumo, pelas perguntas do pesquisador; aparecem só as seções com cálculo (7.2 e 7.4 a 7.14). A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 16; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (3, 7, 12 e 14), o código e o resultado. O código é Python 3 com pandas. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/surgery_cohort.csv`: 2.400 linhas, uma por paciente, e 11 colunas: `patient_id`, `age_years`, `sex`, `diabetes`, `smoker`, `surgery_class` (limpa ou contaminada), `bundle` (pacote de descolonização pré-operatório), `ssi_30d` (infecção de sítio cirúrgico em 30 dias), `complication_30d`, `followup_days` e `readmitted_90d`. Cada célula traz o número da seção do livro a que pertence.

### 7.2 A tabela dois por dois

*Que fração dos pacientes com e sem diabetes teve infecção?*

In [1]:
# 1. Ler o registro cirúrgico: uma linha por paciente
import pandas as pd

df = pd.read_csv("data/surgery_cohort.csv")

# 2. A tabela dois por dois de diabetes (linhas) e infecção em 30 dias (colunas), "yes" primeiro
table = pd.crosstab(df["diabetes"], df["ssi_30d"])
table = table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
print(table)

# 3. O risco de infecção em cada linha: as infecções divididas pelos pacientes da linha
p1 = table.loc["yes", "yes"] / table.loc["yes"].sum()
p0 = table.loc["no", "yes"] / table.loc["no"].sum()
print("Risco com diabetes:", round(p1, 4), "; sem diabetes:", round(p0, 4))

ssi_30d   yes    no
diabetes           
yes        81   519
no        111  1689
Risco com diabetes: 0.135 ; sem diabetes: 0.0617


### 7.4 Diferença de riscos

*Em quanto o diabetes eleva o risco de infecção, em termos absolutos?*

In [2]:
# 1. A diferença de riscos como proporção, em pontos percentuais e por 1.000 pacientes
rd = p1 - p0
print("Diferença de riscos:", round(rd, 4), "=", round(100 * rd, 2), "pontos percentuais =", round(1000 * rd, 1), "por 1.000 pacientes")

Diferença de riscos: 0.0733 = 7.33 pontos percentuais = 73.3 por 1.000 pacientes


### 7.5 Razão de riscos

*Quantas vezes maior é o risco, e o que o pacote faz com ele?*

In [3]:
# 1. A razão de riscos do diabetes e o aumento relativo em %
rr = p1 / p0
print("Razão de riscos do diabetes:", round(rr, 3), "; aumento relativo:", round(100 * (rr - 1), 1), "%")

# 2. O pacote de descolonização: as quatro células (infectados e não infectados, com e sem o pacote)
bundle_table = pd.crosstab(df["bundle"], df["ssi_30d"])
bundle_table = bundle_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
a = int(bundle_table.at["yes", "yes"])
b = int(bundle_table.at["yes", "no"])
c = int(bundle_table.at["no", "yes"])
d = int(bundle_table.at["no", "no"])
print("Células a, b, c, d:", a, b, c, d)

# 3. A razão de riscos do pacote e a redução relativa em %
bundle_p1 = a / (a + b)
bundle_p0 = c / (c + d)
bundle_rr = bundle_p1 / bundle_p0
print("Razão de riscos do pacote:", round(bundle_rr, 3), "; redução relativa:", round(100 * (1 - bundle_rr), 1), "%")

Razão de riscos do diabetes: 2.189 ; aumento relativo: 118.9 %
Células a, b, c, d: 67 1147 125 1061
Razão de riscos do pacote: 0.524 ; redução relativa: 47.6 %


### 7.6 Razão de prevalências

*Em um estudo transversal, como se comparam as prevalências?*

In [4]:
# 1. Prevalência em cada turno de um retrato transversal e a razão de prevalências
night_prevalence = 80 / 400
day_prevalence = 40 / 400
print("Razão de prevalências, noite contra dia:", night_prevalence / day_prevalence)

Razão de prevalências, noite contra dia: 2.0


### 7.7 Razão de chances

*Como se comparam as chances, e quando elas diferem dos riscos?*

In [5]:
# 1. Razão de riscos e razão de chances do diabetes para dois desfechos: infecção e qualquer complicação em 30 dias
for outcome in ["ssi_30d", "complication_30d"]:
    outcome_table = pd.crosstab(df["diabetes"], df[outcome])
    outcome_table = outcome_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    a = int(outcome_table.at["yes", "yes"])
    b = int(outcome_table.at["yes", "no"])
    c = int(outcome_table.at["no", "yes"])
    d = int(outcome_table.at["no", "no"])
    exposed_risk = a / (a + b)
    comparison_risk = c / (c + d)
    risk_ratio = exposed_risk / comparison_risk
    odds_ratio = (a * d) / (b * c)
    print(outcome + ": razão de riscos", round(risk_ratio, 3), "; razão de chances", round(odds_ratio, 3))

ssi_30d: razão de riscos 2.189 ; razão de chances 2.375
complication_30d: razão de riscos 1.847 ; razão de chances 2.882


### 7.8 Razão de taxas

*Como se comparam as taxas de reinternação ao longo do seguimento?*

In [6]:
# 1. Primeiras reinternações e dias de seguimento em cada grupo, e a taxa por 1.000 pacientes-dia
rates = {}
for group in ["yes", "no"]:
    part = df.loc[df["diabetes"] == group]
    events = int(part["readmitted_90d"].eq("yes").sum())
    days = int(part["followup_days"].sum())
    rates[group] = events / days
    print("Diabetes", group + ":", events, "reinternações em", days, "pacientes-dia =", round(1000 * rates[group], 2), "por 1.000 pacientes-dia")

# 2. A razão de taxas
print("Razão de taxas:", round(rates["yes"] / rates["no"], 3))

Diabetes yes: 153 reinternações em 40615 pacientes-dia = 3.77 por 1.000 pacientes-dia
Diabetes no: 253 reinternações em 136144 pacientes-dia = 1.86 por 1.000 pacientes-dia
Razão de taxas: 2.027


### 7.9 Número necessário para tratar e número necessário para causar dano

*Quantos pacientes precisam ser tratados para evitar uma infecção?*

In [7]:
# 1. O pacote: a redução absoluta de risco e o número necessário para tratar (o inverso, arredondado para cima)
import math

bundle_arr = bundle_p0 - bundle_p1
print("Redução absoluta de risco:", round(bundle_arr, 4), "; número necessário para tratar:", math.ceil(1 / bundle_arr))

# 2. Um cenário de menor risco: risco basal de 2%, com a razão de riscos do pacote
scenario_arr = 0.02 * (1 - bundle_rr)
print("Número necessário para tratar com risco basal de 2%:", math.ceil(1 / scenario_arr))

# 3. O exemplo de dano: 20 contra 10 eventos por 1.000 e o número necessário para causar dano
harm_increase = 20 / 1000 - 10 / 1000
print("Número necessário para causar dano:", math.ceil(1 / harm_increase))

Redução absoluta de risco: 0.0502 ; número necessário para tratar: 20
Número necessário para tratar com risco basal de 2%: 105
Número necessário para causar dano: 100


### 7.10 Fração atribuível entre os expostos

*Que fração das infecções entre os expostos é atribuível ao diabetes?*

In [8]:
# 1. Infecções esperadas entre os 600 pacientes com diabetes ao risco dos sem diabetes e o excesso
expected_exposed = 600 * p0
excess_exposed = 81 - expected_exposed
print("Infecções esperadas:", round(expected_exposed), "; infecções em excesso:", round(excess_exposed))

# 2. A fração atribuível entre os expostos
af_exposed = (p1 - p0) / p1
print("Fração atribuível entre os expostos:", round(100 * af_exposed, 1), "%")

Infecções esperadas: 37 ; infecções em excesso: 44
Fração atribuível entre os expostos: 54.3 %


### 7.11 Fração atribuível populacional

*Que fração de todas as infecções é atribuível ao diabetes?*

In [9]:
# 1. A fração atribuível populacional pela comparação direta e pela fórmula com a prevalência da exposição
population_risk = 192 / 2400
exposure_prevalence = 600 / 2400
paf_direct = (population_risk - p0) / population_risk
paf_formula = exposure_prevalence * (rr - 1) / (1 + exposure_prevalence * (rr - 1))
print("Fração atribuível populacional, direta:", round(100 * paf_direct, 1), "%; pela fórmula:", round(100 * paf_formula, 1), "%")

Fração atribuível populacional, direta: 22.9 %; pela fórmula: 22.9 %


### 7.12 Estratificação e confusão

*Como a associação entre tabagismo e infecção se comporta dentro de cada classe de cirurgia?*

In [10]:
# 1. Tabagismo e infecção dentro de cada classe de cirurgia: as quatro células e a razão de riscos da classe
for surgery_class, part in df.groupby("surgery_class", sort=True):
    class_table = pd.crosstab(part["smoker"], part["ssi_30d"])
    class_table = class_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
    a = int(class_table.at["yes", "yes"])
    b = int(class_table.at["yes", "no"])
    c = int(class_table.at["no", "yes"])
    d = int(class_table.at["no", "no"])
    stratum_rr = (a / (a + b)) / (c / (c + d))
    print("Classe", surgery_class + ": células", a, b, c, d, "; razão de riscos", round(stratum_rr, 2))

Classe clean: células 32 570 10 458 ; razão de riscos 2.49
Classe contaminated: células 29 128 121 1052 ; razão de riscos 1.79


### 7.13 Paradoxo de Simpson

*Uma comparação bruta pode se inverter quando a mistura de casos é igualada?*

In [11]:
# 1. Dois hospitais: mortalidade bruta e mortalidade com uma mistura de casos comum (metade graves, metade leves)
for name, severe_n, severe_deaths, mild_n, mild_deaths in [
    ("A", 200, 40, 800, 16),
    ("B", 800, 120, 200, 2),
]:
    crude = (severe_deaths + mild_deaths) / (severe_n + mild_n)
    common_mix = 0.5 * severe_deaths / severe_n + 0.5 * mild_deaths / mild_n
    print("Hospital", name + ": mortalidade bruta", round(100 * crude, 1), "%; com mistura de casos comum", round(100 * common_mix, 1), "%")

Hospital A: mortalidade bruta 5.6 %; com mistura de casos comum 11.0 %
Hospital B: mortalidade bruta 12.2 %; com mistura de casos comum 8.0 %


### 7.14 O que um estudo de caso-controle pode estimar

*O que um estudo de caso-controle pode estimar?*

In [12]:
# 1. Todos os casos de infecção e quatro controles por caso sorteados entre os pacientes sem infecção (semente 7)
cases = df.loc[df["ssi_30d"] == "yes"]
noncases = df.loc[df["ssi_30d"] == "no"]
controls = noncases.sample(n=4 * len(cases), random_state=7)
case_control = pd.concat([cases, controls], ignore_index=True)

# 2. A tabela de diabetes e situação de caso na amostra de caso-controle
cc_table = pd.crosstab(case_control["diabetes"], case_control["ssi_30d"])
cc_table = cc_table.reindex(index=["yes", "no"], columns=["yes", "no"], fill_value=0)
a = int(cc_table.at["yes", "yes"])
b = int(cc_table.at["yes", "no"])
c = int(cc_table.at["no", "yes"])
d = int(cc_table.at["no", "no"])
print("Células a, b, c, d:", a, b, c, d)

# 3. A razão de chances e as frações das linhas (que não são riscos populacionais)
print("Razão de chances:", round(a * d / (b * c), 3))
print("Frações das linhas com diabetes e sem diabetes:", round(a / (a + b), 3), round(c / (c + d), 3))

Células a, b, c, d: 81 179 111 589
Razão de chances: 2.401
Frações das linhas com diabetes e sem diabetes: 0.312 0.159


## Parte B. Soluções dos exercícios

### 7.17 Exercício 1. A tabela da sobremesa

Com a exposição nas linhas e a doença na primeira coluna,
$a=44$, $b=116$, $c=14$, $d=66$. Os totais das linhas são 160 e 80.
Os riscos de ataque são $44/160=27{,}5\%$ e $14/80=17{,}5\%$.
Em contraste, $44/58=75{,}9\%$ é a fração dos convidados doentes que comeram a
sobremesa. Seu denominador contém casos, e não todos os que comeram a sobremesa.

### 7.17 Exercício 2. Lendo uma matriz de confusão

|  | Condição presente | Condição ausente | Total |
|---|---|---|---|
| Teste positivo | 24 (VP) | 17 (FP) | 41 |
| Teste negativo | 6 (FN) | 153 (VN) | 159 |
| Total | 30 | 170 | 200 |


Há 17 falsos positivos e 6 falsos negativos. Entre os 30 pacientes
com a condição, $24/30=80\%$ têm teste positivo. Entre os 41 pacientes com teste
positivo, $24/41=58{,}5\%$ têm a condição. A primeira medida parte da
classificação de referência e a segunda, do resultado do teste.

### 7.17 Exercício 3. Sexo e infecção

As quatro células, com os homens primeiro, são 91, 1.118, 101 e 1.090.
Os homens têm risco de infecção de $91/1.209=7{,}53\%$ e as mulheres,
$101/1.191=8{,}48\%$. A diferença, homens menos mulheres, é
$-0{,}953$ ponto percentual: o risco observado é ligeiramente menor nos homens.
O código a seguir reproduz os riscos e a diferença.
A diferença negativa impressa descreve esta comparação sem estabelecer
que o sexo em si a explique.

In [13]:
# 1. Risco de infecção por sexo, em %: homens e mulheres
sex_table = pd.crosstab(df["sex"], df["ssi_30d"])
sex_table = sex_table.reindex(index=["M", "F"], columns=["yes", "no"])
sex_risks = sex_table["yes"] / sex_table.sum(axis=1)
print("Risco em homens e em mulheres (%):", (100 * sex_risks).round(2).tolist())

# 2. A diferença de riscos entre homens e mulheres, em pontos percentuais
print("Diferença de riscos, homens menos mulheres:", round(100 * (sex_risks["M"] - sex_risks["F"]), 3), "percentage points")

Risco em homens e em mulheres (%): [7.53, 8.48]
Diferença de riscos, homens menos mulheres: -0.953 percentage points


### 7.17 Exercício 4. Razões de risco

Os riscos no ensaio são $100/2.000=0{,}05$ e $150/2.000=0{,}075$:
$\mathrm{RD}=-0{,}025$, $\mathrm{RR}=2/3\approx0{,}667$ e
$\mathrm{RRR}=1/3\approx33{,}3\%$ em dois anos.
Para a sobremesa, $\mathrm{RR}=0{,}275/0{,}175\approx1{,}57$, acima do valor de
1,33 da comparação com os frutos do mar. A sobremesa também tem a maior diferença
absoluta, 10 contra aproximadamente 7 pontos percentuais. Os convidados podem ter
comido os dois pratos; exposição conjunta, momento da ingestão, sintomas e achados
laboratoriais pertinentes são necessários para investigar a causalidade. A maior
razão, isoladamente, não identifica o culpado.

### 7.17 Exercício 5. Prevalência

As prevalências são $90/300=30\%$ e $45/300=15\%$, o que dá RP $=2$ e uma
diferença de 15 pontos percentuais. Uma frase compatível é: “Na avaliação, a dor
lombar estava presente em 30% dos profissionais de enfermagem da enfermaria e em
15% dos do ambulatório, uma razão de prevalências de 2.” O inquérito não
verificou dor nas costas de início recente ao longo do seguimento.

### 7.17 Exercício 6. Risco e chance (odds)

Para riscos de 0,40 e 0,20, RR $=2$ e
OR $=(0{,}40/0{,}60)/(0{,}20/0{,}80)=2{,}667$.
O fator é $0{,}80/0{,}60=1{,}333$, de modo que a OR excede a RR em 33,3%.
Para riscos de 0,04 e 0,02, RR $=2$ e
OR $=(0{,}04/0{,}96)/(0{,}02/0{,}98)=2{,}0417$.
O fator é $0{,}98/0{,}96=1{,}02083$, um excesso relativo de 2,08%. A segunda
aproximação é mais próxima; se essa discrepância é aceitável depende do uso
pretendido.

### 7.17 Exercício 7. Taxas

As taxas de curativo são $30/1.500\times1.000=20$ e
$20/2.500\times1.000=8$ infecções por 1.000 cateteres-dia.
A razão de taxas é $20/8=2{,}5$ e a diferença é de 12 infecções por
1.000 cateteres-dia. Para a comparação do pacote de medidas (*bundle*), o bloco
seguinte soma as reinternações e o tempo em risco em cada grupo.
A razão de taxas observada é 0,85. Nenhum dos dois conjuntos de taxas brutas
estabelece um efeito da intervenção: elegibilidade, prognóstico, seleção e
processos de observação precisam sustentar uma comparação causal.

In [14]:
# 1. Primeiras reinternações e dias de seguimento com e sem o pacote
bundle_rates = {}
for group in ["yes", "no"]:
    part = df.loc[df["bundle"] == group]
    events = int(part["readmitted_90d"].eq("yes").sum())
    days = int(part["followup_days"].sum())
    bundle_rates[group] = events / days
    print("Pacote", group + ":", events, "reinternações em", days, "patient-days")

# 2. A razão de taxas do pacote
print("Razão de taxas do pacote:", round(bundle_rates["yes"] / bundle_rates["no"], 3))

Pacote yes: 192 reinternações em 90760 patient-days
Pacote no: 214 reinternações em 85999 patient-days
Razão de taxas do pacote: 0.85


### 7.17 Exercício 8. Risco basal e NNT

A RAR no ensaio é $0{,}075-0{,}05=0{,}025$, o que dá NNT $=40$ em dois anos.
Mantendo a RR em $2/3$ quando o risco do comparador é 0,02, o risco no tratamento
é $0{,}01\overline{3}$, a RAR é $0{,}00\overline{6}$ e o NNT $=150$.
O segundo cálculo supõe que o mesmo efeito relativo causal se transporta
para a nova população. Sob essa suposição, um risco basal menor dá menos benefício
absoluto.

### 7.17 Exercício 9. Dano

Os riscos são $18/900=0{,}02$ e $6/900=0{,}00\overline{6}$.
O aumento é $12/900=0{,}01\overline{3}$, ou 1,33 ponto percentual.
NND $=900/12=75$ em um ano para sangramento maior. Comparar esse número com
um NNT para acidente vascular cerebral em dois anos exige também comparar os
períodos, as consequências clínicas dos dois desfechos e suas incertezas.

### 7.17 Exercício 10. Fração atribuível nos expostos

O risco nos não expostos é $10/200=0{,}05$. Se ele se aplicasse aos 200 pacientes
expostos com a retirada da exposição, a contagem esperada seria 10 em vez de 30,
uma diferença de 20. A fração nos expostos é $20/30=66{,}7\%$.
Chamar essas infecções de evitáveis afirma uma comparação de intervenção causal:
a retirada especificada daria a esses pacientes o risco de comparação.
A tabela observada, por si só, fornece a associação, e não essa afirmação.

### 7.17 Exercício 11. Fração populacional

A RR na coorte é $(81/600)/(111/1.800)=2{,}\overline{189}$.
Com 10% de prevalência de exposição,

$$ \mathrm{PAF}=\frac{0{,}10(2{,}\overline{189}-1)}{1+0{,}10(2{,}\overline{189}-1)}
=\frac{22}{207}\approx10{,}6\%. $$

A fração populacional é menor que 22,9% porque menos pessoas carregam a
exposição no cenário de efeito fixo. Este cálculo mantém as suposições de
substituição pelo risco dos não expostos e de transporte.

### 7.17 Exercício 12. Estratificando o diabetes

O bloco seguinte calcula os riscos específicos por classe e suas razões, e depois
informa a parcela de operações limpas segundo o status de diabetes.
As contagens de operações limpas são 259 de 600 pacientes com diabetes e 811 de
1.800 sem diabetes. Suas proporções, 43,2% e 45,1%, são muito mais próximas
do que as proporções de tabagismo, 79,3% e 28,5%. As RR dos estratos também
são próximas da RR bruta de 2,19. A concordância não prova a ausência de
confusão por outras variáveis, nem estabelece uma interpretação causal.

In [15]:
# 1. A razão de riscos do diabetes dentro de cada classe de cirurgia
for surgery_class, part in df.groupby("surgery_class", sort=True):
    tab = pd.crosstab(part["diabetes"], part["ssi_30d"])
    risks = tab["yes"] / tab.sum(axis=1)
    print("Classe", surgery_class + ": razão de riscos do diabetes", round(risks["yes"] / risks["no"], 3))

# 2. A porcentagem de operações limpas entre pacientes com e sem diabetes
for group in ["yes", "no"]:
    part = df.loc[df["diabetes"] == group]
    print("Diabetes", group + ": operações limpas", round(100 * part["surgery_class"].eq("clean").mean(), 1), "%")

Classe clean: razão de riscos do diabetes 2.129
Classe contaminated: razão de riscos do diabetes 2.158
Diabetes yes: operações limpas 43.2 %
Diabetes no: operações limpas 45.1 %


### 7.17 Exercício 13. Uma composição de casos hospitalar comum

Usando 25% de pacientes graves e 75% de leves, A tem risco padronizado de
$0{,}25(0{,}20)+0{,}75(0{,}02)=6{,}5\%$ e B tem
$0{,}25(0{,}15)+0{,}75(0{,}01)=4{,}5\%$.
B permanece menor nesta composição comum, embora seu risco bruto observado
tenha sido maior, 12,2% contra 5,6%. As comparações com a composição real e
com pesos comuns respondem a perguntas descritivas diferentes. Categorias amplas
de gravidade ainda podem esconder diferenças prognósticas, e seleção,
verificação dos desfechos e incerteza importam antes de se inferir uma diferença
causal na qualidade do cuidado.

### 7.17 Exercício 14. Dois controles por caso

O bloco seguinte repete o sorteio com uma razão controle-caso diferente.
A OR é 2,130 e não 2,401 porque a amostra aleatória menor de controles
tem uma composição de exposição diferente. As frações das linhas amostradas
sobem para 45,3% e 28,0% porque menos não casos acompanham os mesmos casos.
O alvo populacional da OR permanece o mesmo sob as suposições de amostragem,
mas uma estimativa finita não está algebricamente fixada nesse alvo.

In [16]:
# 1. Todos os casos e dois controles por caso, sorteados entre os pacientes sem infecção (semente 7)
cases = df.loc[df["ssi_30d"] == "yes"]
noncases = df.loc[df["ssi_30d"] == "no"]
controls = noncases.sample(n=2 * len(cases), random_state=7)
selected = pd.concat([cases, controls], ignore_index=True)

# 2. A tabela de diabetes e situação de caso
tab = pd.crosstab(selected["diabetes"], selected["ssi_30d"])
a = int(tab.at["yes", "yes"])
b = int(tab.at["yes", "no"])
c = int(tab.at["no", "yes"])
d = int(tab.at["no", "no"])
print("Células a, b, c, d:", a, b, c, d)

# 3. A razão de chances e as frações das linhas
print("Razão de chances:", round(a * d / (b * c), 3))
print("Frações das linhas com diabetes e sem diabetes:", round(a / (a + b), 3), round(c / (c + d), 3))

Células a, b, c, d: 81 98 111 286
Razão de chances: 2.13
Frações das linhas com diabetes e sem diabetes: 0.453 0.28


### 7.17 Exercício 15. Reescrevendo um benefício relativo

RR $=1-0{,}40=0{,}60$. O outro risco é $0{,}005\times0{,}60=0{,}003$,
de modo que a RAR $=0{,}002$ e o NNT, com interpretação causal, é 500 em dois anos.
Um relatório pode dizer: “O risco de acidente vascular cerebral em dois anos foi
de 0,3% com o novo tratamento e de 0,5% com o comparador, o que corresponde a 3
contra 5 acidentes vasculares cerebrais por 1.000 pacientes, uma redução absoluta
de 0,2 ponto percentual e uma redução relativa de 40%.” As contagens por grupo,
o desenho do estudo e um intervalo de incerteza apropriado são necessários para
avaliar a precisão e a interpretação da estimativa.

### 7.17 Exercício 16. Escolhendo uma medida

(a) Uma redução absoluta do risco com sustentação causal, multiplicada pelo número
de operações elegíveis, estima a mudança esperada na contagem de infecções;
verifique se população, protocolo e horizonte coincidem, e inclua os custos de
implementação.
(b) Uma razão de taxas compara eventos por tempo em risco; examine a censura e se a
observação difere conforme o prognóstico. Se a decisão exige o risco em tempo fixo,
use, em vez disso, uma estimativa de sobrevida adequada.
(c) Uma razão de prevalências e uma diferença de prevalências comparam a hipertensão
existente; verifique se a avaliação e a elegibilidade da população são comparáveis.
(d) Uma fração atribuível populacional descreve a redução proporcional
sob a intervenção especificada; avalie a confusão, a viabilidade da intervenção
e a incerteza, em vez de equiparar associação e prevenção.

---
*Dados sintéticos (`generate_data.py`, semente 7). Valores apenas didáticos; nada aqui é estimativa para um hospital real.*